[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/question-bank/lab-test-co1.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab test — Internship postings

**MSIS · AME 5053 · CO1 · 90 minutes · 30 marks · Open book**

This test is on CO1: text processing for information extraction. It covers Labs 2 and 3.

There is one question with two parts. Both parts use the same data: six internship postings from a
placement cell. You will:

- pull out the details of each posting (Part A);
- clean and tokenise the text (Part B).

Answer both parts.

## Instructions

1. The test runs in Google Colab. First, choose **File → Save a copy in Drive**. Work only in your copy.
   The notebook you were given cannot be saved. Save often with **Ctrl+S**.
2. You may use the course notes, the notebooks for Labs 1–3 and your own notes. You may not talk to anyone
   during the test.
3. Write your answers in the cells marked **Your answer**. You may add more cells if you need them.
4. Use Python's standard library, `regex` and NLTK. These are the libraries from the labs.
5. Use the exact names the question gives, such as `records`, `texts` and `preprocess`.
6. Some parts ask for an explanation. The explanation carries marks. A correct output alone does not get
   full marks.
7. Before you submit, choose **Runtime → Restart session and run all**. Every cell must run without an error.
8. To submit, choose **File → Download → Download .ipynb**. Submit that file.
9. There is a short viva after the test. You must be able to explain every line of your code.

| Part | Lab | Topic | Marks |
|------|-----|-------|-------|
| A | 2 | Extracting the details of each posting | 15 |
| B | 3 | Normalisation, stop words, stemming and lemmatisation | 15 |

**Suggested time.** Part A: 40 minutes. Part B: 40 minutes.
Keep the last 10 minutes to run all cells and check your output.

Now enter your name and registration number below. Then run the two setup cells. The first one takes about
a minute.

In [ ]:
name = ""
registration_number = ""

In [ ]:
# Colab wipes its machine between sessions, so we install at the start of every notebook.
!pip install -q regex nltk

In [ ]:
import datetime
import unicodedata
import nltk
import regex

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

try:
    import google.colab
    print("Setup complete, running on Google Colab")
except ImportError:
    print("Setup complete, but this is not Colab: the test must be run in Google Colab")

---

## The data

Each line has three fields. The fields are separated by `|`.

1. The posting ID, for example `INT-3101`.
2. The date the posting was put up.
3. The posting text.

Look at the data carefully before you start.

- Stipends are written in four ways: `₹25,000`, `Rs. 18,000`, `Rs 15000` and `INR 1,20,000`. The last one
  uses Indian digit grouping.
- Posting 3105 is unpaid. It has no stipend.
- Dates use `-` in some lines and `/` in others.
- Posting 3103 mentions another posting, `INT-3101`, in its text.
- Posting 3104 ends with a sentence in Hindi.

In [ ]:
postings_text = """INT-3101 | 02-10-2026 | Data analyst intern, Bengaluru. Stipend ₹25,000 per month. Clean and analyse sales data using Python and SQL. Apply to careers@brightlytics.example.com by 20/10/2026.
INT-3102 | 03-10-2026 | Backend developer intern, Mysuru. Stipend Rs. 18,000 per month. Build REST services in Java. Apply to jobs@kavericode.example.in by 25-10-2026.
INT-3103 | 05/10/2026 | Machine learning intern, Bengaluru. Stipend INR 1,20,000 for six months. Train and evaluate models in Python; refer to INT-3101 for the data team. Write to ml.hiring@nimbus-ai.example.com by 31/10/2026.
INT-3104 | 06-10-2026 | NLP research intern, Manipal. Stipend Rs 15000 per month. Analysing Hindi and Marathi text with Python. No prior experience required. Contact nlp.lab@ccnlp.example.org by 30-10-2026. हिंदी पढ़ना आना चाहिए।
INT-3105 | 08/10/2026 | Data engineering intern, Mangaluru. Unpaid. Build data pipelines in SQL and Java. No Python required. Apply to talent@coastaldata.example.com by 15/10/2026.
INT-3106 | 09-10-2026 | Analytics intern, Udupi. Stipend ₹12,500 per month. Analyses survey data in Excel and SQL. Apply to hr@udupi-analytics.example.in by 22-10-2026.
"""
print(postings_text)

---

## Part A — The details of each posting (15 marks)

**Step 1.** Write `postings_text` to a file called `internships.txt`. Then read the file back into a list
called `lines`. Skip blank lines. Open the file as UTF-8 both times. (Colab saves the file in `/content`.
It is created again each time you run the notebook.)

**Step 2.** Write a function `extract(line)`. It takes one line and returns a dictionary. Use regular
expressions. The dictionary must have these six keys:

| Key | What to store | Example |
|-----|---------------|---------|
| `id` | The posting number at the **start** of the line, as an `int`. Do not take the number from inside the text. | `3101` |
| `posted` | The first date in the line, as a tuple `(day, month, year)` of `int`s. It may use `-` or `/`. | `(2, 10, 2026)` |
| `deadline` | The date after the word `by`, in the same form. | `(20, 10, 2026)` |
| `city` | The word after the first comma, before the full stop. | `"Bengaluru"` |
| `stipend` | The amount as an `int`, with commas removed. It may start with `₹`, `Rs.`, `Rs` or `INR`. Use `None` if the posting is unpaid. | `25000` |
| `email` | The full email address. It may contain dots and hyphens. | `"careers@brightlytics.example.com"` |

**Step 3.** Call `extract` on every line in `lines`. Keep the lines in the same order. Store the six
dictionaries in a list called `records`. Print each dictionary on its own line.

**Step 4.** For each posting, find how many days it stays open. This is the number of days from `posted` to
`deadline`. Use `datetime.date`. Print the result for each posting.

**Step 5.** Find the postings that are still open on 21 October 2026. A posting is open if its deadline is
on or after that date. Sort them by stipend, highest first. Store their IDs in a list called `still_open`.
Print it.

**Explanation (2–3 sentences).** Posting 3103 is first in `still_open`. Is it really the best-paid
internship? What does your record for 3103 leave out? What extra detail would you need to compare the
stipends fairly?

*Marks:* ID taken from the start of the line (2). Both dates, with either separator, and the deadline taken
after `by` (3). Stipend from all four forms, including Indian grouping, and `None` when unpaid (3). City and
email (2). Days open (1). `still_open` (2). Explanation (2).

In [ ]:
# Your answer to Part A
lines = []

def extract(line):
    # Replace this line with your implementation.
    return {}

records = []
still_open = []

# Write your code here.

**Explanation:** Write your answer here.

---

## Part B — Cleaning and tokenising the text (15 marks)

From now on we use only the posting text. This is the third field of each line.

**Step 1.** Make a dictionary called `texts`. Its keys are the posting numbers, such as `3101`. Its values
are the posting texts.

**Step 2.** Write a function `preprocess(text, mode="stem")`. It returns a list of tokens. It must do these
five things, in this order:

1. Apply NFC Unicode normalisation.
2. Remove three things from the text: every email address, every amount with its currency sign (such as
   `₹25,000` or `Rs. 18,000`), and every posting ID (`INT-` followed by four digits).
3. Change the text to lower case. Then split it into tokens with the `regex` package. A token may contain
   only Unicode letters and combining marks. So digits are dropped, and the Hindi words stay whole.
4. Remove NLTK's English stop words. But keep `no` and `not`. Do not remove these two.
5. Check `mode`:
   - If `mode` is `"stem"`, stem every token with the Porter stemmer.
   - If `mode` is `"lemma"`, lemmatise every token with the WordNet lemmatiser. Use its default settings.
   - If `mode` is `None`, leave the tokens as they are.

**Step 3.** For every posting, print the posting number and its stemmed tokens.

**Step 4.** Count the vocabulary size of all six postings together. Do this three times: with `mode=None`,
with `mode="stem"` and with `mode="lemma"`. Print the three sizes.

**Step 5.** Take these five words: `analyst`, `analyse`, `analysing`, `analyses`, `analytics`. For each
word, print the word, its stem and its lemma on one line.

**Explanation (3–4 sentences).** Look at `analyse`, `analysing` and `analyses`. Which mode turns all three
into the same term? Why does the lemmatiser not do this, the way you used it? Also give one token from your
stemmed output that is not a real English word.

*Marks:* NFC and the three removals (3). Tokens that keep the Hindi words whole (2). Stop words removed, with
`no` and `not` kept (2). All three modes (3). The three vocabulary sizes and the word table (2).
Explanation (3).

In [ ]:
# Your answer to Part B
texts = {}

def preprocess(text, mode="stem"):
    # Replace this line with your implementation.
    return []

# Apply it and print the results here.

**Explanation:** Write your answer here.

---

## Before you submit

1. Choose **Runtime → Restart session and run all**. Wait for the last cell to finish.
2. Check that your name and registration number are filled in.
3. Check that every result the question asks for is printed.
4. Check that no cell shows an error.
5. Choose **File → Download → Download .ipynb**. Submit that file.